## Install Libraries

In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q pypdf
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q accelerate
!pip install -q ragas
!pip install -q deepeval
!pip install -q presidio-analyzer presidio-anonymizer

## Import Libraries

In [9]:
import langchain
print(langchain.__version__)

1.3.14


In [10]:
!pip list | findstr langchain

langchain                                1.3.14
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.5.1
langchain-openai                         1.4.1
langchain-protocol                       0.0.18
langchain-text-splitters                 1.1.2


In [12]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-huggingface
!pip install -U pypdf
!pip install -U faiss-cpu
!pip install -U sentence-transformers

In [18]:
!pip show langchain
!pip show langchain-community
!pip show langchain-core
!pip show langchain-text-splitters
!pip show langchain-huggingface

Name: langchain
Version: 1.3.14
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\dell\anaconda3\Anaconda\envs\myenv6\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: ragas
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\dell\anaconda3\Anaconda\envs\myenv6\Lib\site-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: ragas
Name: langchain-core
Version: 1.5.1
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\dell\anaconda3\Anaconda\envs\myenv6\Lib\site-packages
Requires: jsonpatch, langchain-protoco

In [19]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-core
!pip install -U langchain-text-splitters
!pip install -U langchain-huggingface
!pip install -U pypdf
!pip install -U faiss-cpu
!pip install -U sentence-transformers

In [1]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import pipeline

C:\Users\dell\AppData\Local\Temp\ipykernel_9980\1925391740.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\dell\anaconda3\Anaconda\envs\myenv6\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load PDF

In [2]:
loader = PyPDFLoader("law.pdf")

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 4


## Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

print("Chunks:", len(docs))

Chunks: 16


## Generate Embeddings

In [4]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## Create Vector Database

In [5]:
db = FAISS.from_documents(docs, embedding)

db.save_local("vector_db")

## Load Vector DB

In [6]:
db = FAISS.load_local(
    "vector_db",
    embedding,
    allow_dangerous_deserialization=True
)

## Retriever

In [7]:
retriever = db.as_retriever(
    search_kwargs={"k":3}
)

## Load LLM

In [8]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

c:\Users\dell\anaconda3\Anaconda\envs\myenv6\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dell\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu


## Input Guardrails

In [9]:
blocked_words = [
    "hack",
    "password",
    "ignore previous instructions",
    "jailbreak",
    "bypass"
]

def input_guardrail(question):

    for word in blocked_words:
        if word.lower() in question.lower():
            return False

    return True

## Retrieve Context

In [10]:
def retrieve(question):

    docs = retriever.invoke(question)

    context = "\n".join([d.page_content for d in docs])

    return context

## Generate Response

In [11]:
def generate(question):

    context = retrieve(question)

    prompt = f"""
Answer only using the context.

Context:
{context}

Question:
{question}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=256
    )

    return result[0]["generated_text"]

## Output Guardrails

In [12]:
def output_guardrail(answer):

    banned = [
        "hate",
        "violence",
        "kill"
    ]

    for word in banned:
        if word.lower() in answer.lower():
            return "Blocked"

    return "Safe"

## Evaluation

In [13]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

def semantic_score(question, answer):

    q = model.encode([question])

    a = model.encode([answer])

    score = cosine_similarity(q, a)[0][0]

    return score

## Main Chat Function

In [14]:
def chat(question):

    if not input_guardrail(question):
        return "Unsafe Question"

    answer = generate(question)

    status = output_guardrail(answer)

    score = semantic_score(question, answer)

    print("Answer:\n")
    print(answer)

    print("\nSafety:", status)

    print("Semantic Score:", round(score,3))

## Test

In [16]:
chat("What is this document about?")

Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors


Answer:

To describe the Employees Provident Fund Scheme Form 14 (Paragraph 62 of the Employees’ Provident Funds Scheme, 1952) Application for Financing a Life Insurance Policy out of the Provident Fund Account To, The Commissioner, Employees’ Provident Fund, ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... .. I ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... .. I ... ... ... ... ... ... ... ... ... ... ... ... ... ..

Safety: Safe
Semantic Score: 0.294
